# Meetup 4 Guide (Team 1 - World Stock Prices): Shifting Your Target, Class Imbalance & Building Your Baseline

Welcome! Based on what is currently **pushed to GitHub**, you already collected 26 years of stock CSVs with `growth_prct` and 4-class `target` (`0-3`), while your project notebooks are still empty.
*(If you already ran models locally on your laptop, awesome - just commit and push them, and jump straight to the parts you need!)*

### How to Use This Guide
This notebook does **not** solve your project for you. Instead, it explains **each concept step-by-step on tiny toy examples** (just a few lines of code at a time) so everyone on the team understands *what* the code does and *why* - and then gives you a clear **Your Turn** checklist to build `notebooks/02_baseline.ipynb` on your real CSVs.

In [ ]:
# Run once if any library is missing in your VS Code / Jupyter kernel
%pip install -q numpy pandas matplotlib seaborn scikit-learn

## Concept 1: Fixing Same-Day Target Leakage (`groupby` + `shift(-1)`)

### Why does this matter for your repo?
In the CSVs currently on GitHub, `growth_prct` and `target` sit on the **same row** as that day's `Open` and `Close`.
If you ask a model to predict `target` from that same row, it does not learn to forecast **tomorrow** - it just learns a math formula on **today's** numbers!

### What does `shift(-1)` do?
`shift(-1)` pulls **tomorrow's** target UP into **today's** row. Now:
*   Row `Monday` holds **Monday's features** (`X`) and **Tuesday's class** (`y`).
*   Why `groupby('Ticker')`? Because your table stacks many companies together. Grouping keeps each company's timeline separate so the last day of `AAPL` never leaks into the first day of `MSFT`!

Look at this tiny 6-row example to see it clearly:

In [ ]:
import pandas as pd

# Tiny 6-row toy table (2 companies, 3 days each)
toy = pd.DataFrame({
    'Date':   ['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-01', '2024-01-02', '2024-01-03'],
    'Ticker': ['AAPL', 'AAPL', 'AAPL', 'MSFT', 'MSFT', 'MSFT'],
    'Close':  [100, 102, 99, 200, 210, 208],
    'target': [2, 2, 1, 2, 2, 1]  # Same-day class (DO NOT use directly as y!)
})

# 1. Always sort by Ticker and Date first
toy = toy.sort_values(['Ticker', 'Date'])

# 2. Shift target UP by 1 day INSIDE each company -> now row t predicts day t+1!
toy['target_next_day'] = toy.groupby('Ticker')['target'].shift(-1)

display(toy)

Notice the `NaN` on the last day (`2024-01-03`) of each company - that is **good**! You do not know what happened on `2024-01-04`, so you simply drop those `NaN` rows with `.dropna()`.

---
## Concept 2: Class Imbalance, `class_weight='balanced'`, and `Macro F1`

### What is the problem with rare classes (like your Class 0 and Class 3)?
On daily stock prices, `+/- 10%` moves are very rare (for example, in 2023 Class 3 had only 15 rows out of ~15,000).
*   If you train a normal model, it learns a lazy shortcut: *"Never predict Class 0 or Class 3, just guess Class 1 or 2!"*
*   Plain **`Accuracy`** hides this failure (you still get ~50% accuracy while catching 0% of crashes or surges).

### How do you handle it?
1.  **Check `value_counts()` first!** Combining multiple years (e.g., 2018-2025) gives you more Class 0 and Class 3 examples. You can also discuss as a team whether to keep `+/- 10%` or tighten thresholds (like `+/- 3%` or 5-day returns).
2.  **Use `class_weight='balanced'`:** Tells the model that missing a rare class costs much more than missing a common class.
3.  **Use `Macro F1`:** Averages the F1 score of all 4 classes equally, so ignoring rare classes is immediately exposed!

Here is how a **chronological split** + **`Pipeline`** + **`DummyClassifier`** vs. **`LogisticRegression`** works in just a few lines on tiny toy data:

In [ ]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Tiny synthetic time series (200 days, 2 simple features, 4 imbalanced classes)
np.random.seed(42)
n = 200
X_toy = pd.DataFrame({
    'ret_1d': np.random.normal(0, 3, n),   # Today's % return (known at close)
    'ret_5d': np.random.normal(0, 6, n)    # Past 5-day % return
})
# Imbalanced classes (most days are 1 or 2; few days are 0 or 3)
y_toy = np.random.choice([0, 1, 2, 3], size=n, p=[0.05, 0.45, 0.45, 0.05])

# CHRONOLOGICAL 80/20 SPLIT (first 160 rows = past train, last 40 rows = future test)
cut = int(len(X_toy) * 0.8)
X_train, y_train = X_toy.iloc[:cut], y_toy[:cut]
X_test,  y_test  = X_toy.iloc[cut:], y_toy[cut:]

# 1. Dumb Benchmark: always guesses the single most frequent class
dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
pred_dummy = dummy.predict(X_test)

# 2. First Simple Baseline inside a Pipeline (scales features ONLY on X_train -> zero leakage!)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])
pipe.fit(X_train, y_train)
pred_lr = pipe.predict(X_test)

print(f"Dummy  -> Accuracy: {accuracy_score(y_test, pred_dummy):.3f} | Macro F1: {f1_score(y_test, pred_dummy, average='macro'):.3f}")
print(f"LogReg -> Accuracy: {accuracy_score(y_test, pred_lr):.3f} | Macro F1: {f1_score(y_test, pred_lr, average='macro'):.3f}")

## Concept 3 (When Your Baseline Works): Rolling Features & `TimeSeriesSplit`

Once your Baseline runs on your real data, how do you improve it?
1.  **Engineer backward-looking features** (strictly past data!). For example, **rolling volatility** (`rolling(10).std()`) tells the model how wildly a stock has been moving lately - super relevant for spotting `+/- 10%` days!
2.  **Validate with `TimeSeriesSplit(n_splits=5)`** instead of relying on one single 80/20 split.
3.  **Try a stronger model** like `RandomForestClassifier` or `HistGradientBoostingClassifier` and inspect the **Confusion Matrix** to see which classes get confused.

Here is the tiny 3-line pattern for adding a rolling feature safely inside `groupby('Ticker')`:

In [ ]:
# Pattern: computing a backward-looking rolling feature safely inside each company
# (Notice .rolling(window) only looks backward at past rows -> zero future leakage!)
toy_roll = pd.DataFrame({
    'Ticker': ['AAPL'] * 6,
    'ret_1d': [1.2, -0.5, 2.1, -1.8, 0.9, 3.4]
})
toy_roll['volatility_3d'] = toy_roll.groupby('Ticker')['ret_1d'].transform(lambda s: s.rolling(3).std())
display(toy_roll)

---
## Your Turn! (Today or Before Next Week's Meeting)

Create **`notebooks/02_baseline.ipynb`** in your repository and build the real version on your own stock CSVs!

### Checklist
*   **BASIC:**
    1. Load several years of your processed CSVs (e.g., 2020-2025) and convert `Date` with `pd.to_datetime`.
    2. Sort by `['Ticker', 'Date']` and create `target_next_day` using `groupby('Ticker')['target'].shift(-1)`.
    3. Print `value_counts()` for `target_next_day`. *(Discuss as a team: are Classes 0 and 3 large enough, or do you want to adjust thresholds / predict 5-day change?)*
    4. Do a **chronological 80/20 split** and compare `DummyClassifier` vs. `LogisticRegression` vs. `RandomForestClassifier` inside a `Pipeline` (remember to include `Ticker` via `OneHotEncoder` if you want company info!).
    5. Log `Macro F1`, `Accuracy`, and plot a 4x4 **Confusion Matrix** (`ConfusionMatrixDisplay.from_predictions`).
*   **PRO (if you finish early or before next week):**
    1. Design and code **2-3 of your own financial features** (ideas: rolling volatility, `Close / rolling_mean(20) - 1`, volume ratio, or RSI).
    2. Evaluate your Baseline vs. Engineered Features using **`TimeSeriesSplit(n_splits=5)`** and test **`HistGradientBoostingClassifier`**.

*(Remember to commit and push your notebook to GitHub!)*

In [ ]:
# STARTER SKELETON FOR YOUR notebooks/02_baseline.ipynb (Fill in the TODOs!)

# TODO 1: Load your real CSVs from data/processed/ (try combining multiple years!)
# ...

# TODO 2: Sort by ['Ticker', 'Date'], create backward-looking features (e.g., ret_1d, ret_5d),
#         and shift your target UP by 1 day: df.groupby('Ticker')['target'].shift(-1)
# ...

# TODO 3: Drop NaNs, print target_next_day.value_counts(), and split chronologically (oldest 80% / newest 20%)
# ...

# TODO 4: Build a Pipeline and compare DummyClassifier vs. LogisticRegression vs. RandomForestClassifier
# ...

# TODO 5: Print Macro F1 + plot the 4x4 Confusion Matrix
# ...